<a href="https://colab.research.google.com/github/ramyajeldy/Capstone_V2/blob/feature%2Fdataset/BERT_phishingmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

!pip install evaluate # Install the missing library
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
dataset = load_dataset("drorrabin/phishing_emails-data")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/11.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26946 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3705 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})

In [3]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})
['text', 'email_type']
{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.c

In [4]:
dataset["train"].to_pandas()["email_type"].value_counts()

,count
email_type,
phishing email,13473
safe email,13473


In [5]:
def encode_labels(example):
    if example["email_type"] == "phishing email":
        example["labels"] = 1
    else:
        example["labels"] = 0
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

In [6]:
dataset = dataset.remove_columns(["email_type"])

In [7]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

In [9]:

tokenized_dataset.set_format("torch")

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

In [12]:
set(dataset["train"]["labels"])

{0, 1}

In [13]:
tokenized_dataset.set_format("torch")

In [14]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

In [15]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=predictions, references=labels)["f1"],
        "precision": metric_precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": metric_recall.compute(predictions=predictions, references=labels)["recall"],
    }

In [16]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="tensorboard"   # ✅ new way
)

In [17]:
torch.cuda.is_available()

True

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.004071,0.073218,0.977868,0.861017,1.000000,0.755952
2,0.007781,0.025175,0.992443,0.957576,0.975309,0.940476
3,0.001094,0.069220,0.986235,0.918138,0.996516,0.851190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5055, training_loss=0.008325685847239268, metrics={'train_runtime': 4451.5818, 'train_samples_per_second': 18.159, 'train_steps_per_second': 1.136, 'total_flos': 1.063468574659584e+16, 'train_loss': 0.008325685847239268, 'epoch': 3.0})